In [2]:
import pandas as pd
import pyodbc
from datetime import timedelta
import re

def generate_llm_friendly_report(db_path, target_fund_code, start_date, end_date, output_md_path, perf_df):
    """
    极致 Token 压缩版报告生成器：
    1. 排名分位数化：将 "898/4814" 转换为 "18.6%"。
    2. 指标释义：在报告顶部增加 Legend 帮助 LLM 理解。
    3. 表格化输出：仅保留核心排名百分比。
    """

    # --- 指标解释手册 (Legend) ---
    METRIC_LEGEND = {
        "AvgReturn": "平均收益率排名 (越小越优, 0%代表同类第一)",
        "StdReturn": "波动率排名 (越小代表波动越小/越稳)",
        "Sharpe": "夏普比率排名 (风险收益比, 越小代表单位风险收益越高)",
        "Alpha": "超额收益排名 (选股带来的超额回报)",
        "Treynor": "特雷诺比率排名 (系统性风险调整后收益)",
        "TMTiming": "TM模型-择时能力排名",
        "TMStkSlct": "TM模型-选股能力排名"
    }

    def convert_to_percent(rank_str):
        """将 '898/4814' 转换为百分比字符串"""
        if pd.isna(rank_str) or not isinstance(rank_str, str) or '/' not in rank_str:
            return "-"
        try:
            parts = rank_str.split('/')
            rank = float(parts[0])
            total = float(parts[1])
            if total == 0: return "-"
            # 计算百分比并保留1位小数，去掉百分号以节省更多token（在表头说明即可）
            return f"{round((rank / total) * 100, 1)}%"
        except:
            return "-"

    conn_str = (
        r'DRIVER={Microsoft Access Driver (*.mdb, *.accdb)};'
        rf'DBQ={db_path};'
    )

    try:
        conn = pyodbc.connect(conn_str)
        df_all = pd.read_sql("SELECT * FROM [FundManager]", conn)
        conn.close()
        df_all.columns = df_all.columns.str.strip()
    except Exception as e:
        print(f"数据库读取失败: {e}")
        return

    # 数据标准化
    df_all['MasterFundCode'] = df_all['MasterFundCode'].astype(str).str.split('.').str[0].str.zfill(6)
    target_fund_code_clean = str(target_fund_code).split('.')[0].zfill(6)
    df_all['ServiceStartDate_DT'] = pd.to_datetime(df_all['ServiceStartDate'], errors='coerce')
    df_all['ServiceEndDate_DT'] = pd.to_datetime(df_all['ServiceEndDate'], errors='coerce')

    t_start = pd.to_datetime(start_date)
    t_end = pd.to_datetime(end_date)

    # 锁定经理
    mask = (df_all['MasterFundCode'] == target_fund_code_clean) & \
           (df_all['ServiceStartDate_DT'] <= t_end) & \
           (df_all['ServiceEndDate_DT'].fillna(pd.Timestamp('2099-12-31')) >= t_start)

    target_ids = df_all[mask]['FundManagerID'].unique()

    if len(target_ids) == 0:
        print(f"未找到基金 {target_fund_code_clean} 的经理记录。")
        return

    with open(output_md_path, 'w', encoding='utf-8') as f:
        f.write(f"# FUND_ANALYSIS_REPORT: {target_fund_code_clean}\n\n")

        # 写入指标释义，帮助 LLM 理解百分比含义
        f.write("### [METRIC_LEGEND]\n")
        f.write("> Values below are Percentile Ranks (e.g., 10% = Top 10% in category). Lower is better.\n")
        for k, v in METRIC_LEGEND.items():
            f.write(f"- {k}: {v}\n")
        f.write("\n")

        for m_id in target_ids:
            m_records = df_all[df_all['FundManagerID'] == m_id]
            m_info = m_records.sort_values('ServiceStartDate', ascending=False).iloc[0]

            f.write(f"## === MANAGER_START: {m_info['ManagerName']} ({m_id}) ===\n")
            f.write(f"### [PROFILE]\n- Degree: {m_info.get('Degree', 'N/A')} | Univ: {m_info.get('University', 'N/A')} | Major: {m_info.get('Major', 'N/A')}\n")
            f.write(f"### [RESUME]\n{m_info.get('Resume', 'N/A')[:400]}...\n\n")

            f.write("### [PERFORMANCE_PERCENTILE_TABLE]\n")
            other_managed = m_records[m_records['MasterFundCode'] != target_fund_code_clean]

            if other_managed.empty:
                f.write("无其他历史管理记录。\n")
            else:
                for _, row in other_managed.sort_values('ServiceStartDate', ascending=False).iterrows():
                    f_code = row['MasterFundCode']
                    f_name = row.get('FundName', 'Unknown')
                    f.write(f"#### Fund: {f_code} ({f_name})\n")

                    sub_perf = perf_df[perf_df['Symbol'] == f_code].copy()
                    if not sub_perf.empty:
                        sub_perf['TradingDate_DT'] = pd.to_datetime(sub_perf['TradingDate'])
                        latest_date = sub_perf['TradingDate_DT'].max()
                        recent_year = sub_perf[sub_perf['TradingDate_DT'] >= (latest_date - timedelta(days=365))]
                        recent_year = recent_year.sort_values('TradingDate', ascending=False)

                        if not recent_year.empty:
                            # 仅选择含有排名信息的列
                            rank_cols = [c for c in recent_year.columns if '_FullRnk' in c or '_Rnk' in c]

                            if rank_cols:
                                # 简化表头
                                display_headers = [c.replace('_FullRnk', '').replace('_Rnk', '') for c in rank_cols]
                                f.write("| Date | " + " | ".join(display_headers) + " |\n")
                                f.write("| :--- | " + " | ".join([":---" for _ in display_headers]) + " |\n")

                                for _, p_row in recent_year.iterrows():
                                    # 将原始排名字符串转换为百分比
                                    percent_vals = [convert_to_percent(p_row[c]) for c in rank_cols]
                                    f.write(f"| {p_row['TradingDate']} | {' | '.join(percent_vals)} |\n")
                            else:
                                f.write("- 无可用排名数据。\n")
                        else:
                            f.write("- 近12个月无数据。\n")
                    f.write("\n")

            f.write(f"## === MANAGER_END ===\n\n")

In [3]:
# ==========================================
# 执行入口
# ==========================================
if __name__ == "__main__":
    # 更新后的路径
    ACCESS_DB = r"C:\Users\qyw\Desktop\智能体\各智能体\基金经理分析智能体\数据\处理后\Fundmanagerinfo.accdb"
    PERFORMANCE_CSV = r"D:\数据\业绩\处理后\FundPerform.csv"

    print("正在加载业绩大表并清洗代码格式...")
    all_perf_df = pd.read_csv(PERFORMANCE_CSV, encoding='utf-8-sig', low_memory=False)
    # 清洗业绩表的基金代码
    all_perf_df['Symbol'] = (
        all_perf_df['Symbol']
        .astype(str)
        .str.replace(r'\.0$', '', regex=True)
        .str.zfill(6)
    )
    print(f"业绩表加载完成，共 {len(all_perf_df)} 条记录。")

    # 配置目标参数
    TARGET_FUNDS = ["000005"] # 可在此列表中添加多个基金代码
    START_DATE = "2023-01-01"
    END_DATE = "2025-12-31"

    for code in TARGET_FUNDS:
        output_name = f"{code}_Manager_Report.md"
        print(f"正在为基金 {code} 生成穿透分析报告...")
        generate_llm_friendly_report(
            db_path=ACCESS_DB,
            target_fund_code=code,
            start_date=START_DATE,
            end_date=END_DATE,
            output_md_path=output_name,
            perf_df=all_perf_df
        )

    print("\n[完成] 报告已保存至当前目录下。")

正在加载业绩大表并清洗代码格式...
业绩表加载完成，共 1629708 条记录。
正在为基金 000005 生成穿透分析报告...


C:\Users\qyw\AppData\Local\Temp\ipykernel_5068\3640879480.py:46: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_all = pd.read_sql("SELECT * FROM [FundManager]", conn)



[完成] 报告已保存至当前目录下。
